In [ ]:
import torch
import torchrl.data import ReplayBuffer
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import stable_retro
import matplotlib.pyplot as plt
import numpy as np
import gymnasium as gym
import os
from datetime import datetime

#### Mario Environment and Rewards

In [ ]:
def compute_reward(prev_info, curr_info, done):
    reward = 0.0

    # --- Milestones ---
    if curr_info.get('midpoint_flag', 0) > prev_info.get('midpoint_flag', 0):
        reward += 5.0

    if curr_info.get('game_mode', 20) == 12 and prev_info.get('game_mode', 20) == 20:
        reward += 10.0

    # --- Powerups ---
    if curr_info.get('powerup_status', 0) > prev_info.get('powerup_status', 0):
        reward += 1.0

    # reward -= 0.001

    return reward


In [ ]:
ACTIONS = [
    [0,0,0,0,0,0,0,0,0,0,0,0],  # nothing
    [0,0,0,0,0,0,0,1,0,0,0,0],  # right
    [1,0,0,0,0,0,0,1,0,0,0,0],  # right + B (jump)
    [0,1,0,0,0,0,0,1,0,0,0,0],  # right + Y (run)
    [1,1,0,0,0,0,0,1,0,0,0,0],  # right + B + Y (run jump)
    [1,0,0,0,0,0,0,0,0,0,0,0],  # B (jump in place)
    [0,0,0,0,0,0,1,0,0,0,0,0],  # left
    [1,0,0,0,0,0,1,0,0,0,0,0],  # left + B (jump left)
    [0,1,0,0,0,0,1,0,0,0,0,0],  # left + Y (run)
    [0,0,0,0,1,0,0,0,0,0,0,0],  # up
    [0,0,0,0,0,1,0,0,0,0,0,0],  # down
]

class SMWEnv(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        # Override the observation space to match your tuple state
        self.actions = ACTIONS
        self.observation_space = gym.spaces.Box(
            low=np.array([0, 0, 0, 0, -128, 0], dtype=np.float32),
            high=np.array([255, 9999, 255, 999, 128, 1], dtype=np.float32),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Discrete(len(ACTIONS))
        self.stuck_counter = 0
        self.last_x = 0
        self.max_x_reached = 0
        self.prev_info = {}

    def reset(self, **kwargs):
        _, info = self.env.reset(**kwargs)
        self.prev_info = {'x': 0, 'midpoint_flag': 0, 'lives': 4, 'game_mode': 20}
        self.max_x_reached = 0
        obs = self.get_obs(info)
        return obs, info

    def step(self, action):
        actual_action = self.actions[action]
        _, _, terminated, truncated, info = self.env.step(actual_action)
        obs = self.get_obs(info)
        reward = compute_reward(self.prev_info, info, terminated or truncated)
        curr_x = info.get('x', 0)
        curr_pos = info.get('screen', 0) * 256 + curr_x
        if curr_pos > self.max_x_reached:
            reward += (curr_pos - self.max_x_reached) * 0.02
            self.max_x_reached = curr_pos

        curr_time = info.get('timer_h', 0) * 100 + info.get('timer_t', 0) * 10 + info.get('timer_o', 0)
        if curr_time == 0:
            terminated = True
            reward -= 3.0
        if info.get('game_mode', 20) == 11 and self.prev_info.get('game_mode', 20) == 20:
            terminated = True
            reward -= 3.0
        if info.get('game_mode', 20) == 12 and self.prev_info.get('game_mode', 20) == 20:
            terminated = True


        if curr_x == self.last_x:
            self.stuck_counter += 1
        else:
            self.stuck_counter = 0
            self.last_x = curr_x

        if self.stuck_counter > 600:
            terminated = True
            reward -= 4.5
            self.stuck_counter = 0

        self.prev_info = info
        return obs, reward, terminated, truncated, info

    def get_obs(self, info):
        curr_time = info.get('timer_h', 0) * 100 + info.get('timer_t', 0) * 10 + info.get('timer_o', 0)
        mario_x = info.get('x', 0)
        mario_x_local = mario_x % 256
        if info.get('sprite1_x', 0) == 0:
            sprite1_dist = 0.0  # no sprite, neutral signal
        else:
            sprite1_dist = float(np.clip(info.get('sprite1_x', 0) - mario_x_local, -128, 128))
        return np.array([
            info.get('screen', 0),
            mario_x,
            info.get('y', 0),
            curr_time,
            sprite1_dist,
            info.get('midpoint_flag', 0),
        ], dtype=np.float32)

#### JERK Algorithm

In [ ]:
try:
    base_env.close()
except:
    pass

In [ ]:
base_env = stable_retro.make('SuperMarioWorld-Snes-v0', render_mode=None)
# base_env = stable_retro.make('SuperMarioWorld-Snes-v0', render_mode='human')
env = SMWEnv(base_env)

total_rewards = []
steps_per_ep  = []
max_x_per_ep  = []
actor_losses  = []
critic_losses = []

In [ ]:
class JERK:
    def __init__(self, beta, jump_times, jump_presses, right_times, left_times, steps):
        self.beta = beta
        self.jump_times = jump_times
        self.jump_presses = jump_presses
        self.right_times = right_times
        self.left_times = left_times
        self.steps = steps

        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        self.values = []
        self.dones = []

        self.S = {}
        self.T = 0

        self.finish = True
        self.ep_complete = True
        self.curr_steps = 0

    def collect_traject(self):
        for _ in range(self.num_steps):
            
            #set up my actor and critic
            action, log_prob = self.actor(torch.tensor(self.state, dtype=torch.float32))
            action_idx = action.item()
            next_state, reward, terminated, truncated, info = self.env.step(action_idx)
            done = terminated or truncated
            value = self.critic(torch.tensor(self.state, dtype=torch.float32))


            #grab all my trajectories and append them
            self.states.append(self.state)
            self.actions.append(action_idx)
            self.rewards.append(reward)
            self.log_probs.append(log_prob)
            self.values.append(value)
            self.dones.append(done)

            # reset if episode ended
            if done:
                self.state, _ = self.env.reset()
            else:
                self.state = next_state

        #return my set of trajectories
        return self.states, self.actions, self.rewards, self.log_probs, self.values, self.dones

    def rollout(self):
        while not self.finish:
            self.curr_steps += 1
            if abs(self.S) and np.random.uniform(0,1) < self.beta + self.T/self.steps:
                trajcts = self.collect_traject()
                best_traject = ReplayBuffer(trajcts)
                self.rewards.append(best_traject)
                T = self.steps - self.curr_steps 
            else:
                while not self.ep_complete:
                    actions = self.actions[self.right_times, self.jump_times, self.jump_presses]
                    SMWEnv.step(self, actions)
                    if self.rewards == self.rewards:
                        SMWEnv.step(self, self.left_times)
                    T = self.steps - self.curr_steps
                self.ep_complete = False
            r = self.rewards[self.steps]
            self.S.append(best_traject, r)
            if T >= self.steps:
                self.finish = False

In [ ]:
agent = JERK()
agent.rollout()